## Install Packages

In [43]:
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.vectorstores import Chroma


from langchain_core.output_parsers import StrOutputParser
from langchain_community.chat_models import ChatOllama
# # from langchain_core.runnables import RunnablePassthrough
# # from langchain.retrievers.multi_query import MultiQueryRetriever

from langchain.chains import ConversationalRetrievalChain  
# import json

 
# from dotenv import load_dotenv,find_dotenv
# from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory, ChatMessageHistory
# from langchain.prompts import PromptTemplate

from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
# # from langchain_community.vectorstores import Chroma



# load_dotenv(find_dotenv())

# from langchain_community.document_loaders import UnstructuredPDFLoader
# from langchain_community.document_loaders import OnlinePDFLoader

In [21]:
from langchain.document_loaders import PyPDFLoader

pdf_path = "/home/abusufyan/development/local_llm/data/Abu_Sufyan_Bio.pdf"

# Create an instance of the PyPDFLoader with the PDF file path
loader = PyPDFLoader(pdf_path)

# Load the documents from the PDF
data = loader.load()


In [24]:
# Preview first page
data[0].page_content

'Biography of Abu Sufyan\nMy name is Abu Sufyan. I am the son of Zafar Iqbal, and I am 25 years old. I graduated from the\nVirtual University of Pakistan with a degree in Computer Science. During my studies, I specialized in\nthe field of computer vision, and for my final year project, I worked on brain tumor segmentation,\nwhich was an enriching and challenging experience. Currently, I reside in Lahore, but my hometown\nis Hafizabad City, located in the province of Punjab. Growing up in Hafizabad, I developed a deep\nappreciation for my roots and the culture of Punjab, which continues to influence my life and work.'

## Pull Vector Embeddings

In [27]:
!ollama pull nomic-embed-text

pulling manifest ⠙ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠼ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠙ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠇ pulling manifest ⠏ pulling manifest 
pulling 970aa74c0a90... 100% ▕████████████████▏ 274 MB                         
pulling c71d239df917... 100% ▕████████████████▏  11 KB                         
pulling ce4a164fc046... 100% ▕████████████████▏   17 B                         
pulling 31df23ea7daa... 100% ▕████████████████▏  420 B                         
verifying sha256 digest ⠋ pulling manifest 
pulling 970aa74c0a90... 100% ▕████████████████▏ 274 MB                         
pulling c71d239df917... 100% ▕████████████████▏  11 KB                         
pulling ce4a164fc046... 100% ▕████████████████▏   17 B                         
pulling 31df23ea7

In [28]:
!ollama list

NAME                   	ID          	SIZE  	MODIFIED      
nomic-embed-text:latest	0a109f422b47	274 MB	6 seconds ago	
mistral:latest         	61e88e884507	4.1 GB	6 days ago   	
mistral:instruct       	61e88e884507	4.1 GB	6 days ago   	


In [29]:
# Split and chunk 
# text_splitter = RecursiveCharacterTextSplitter(chunk_size=7500, chunk_overlap=100)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = text_splitter.split_documents(data)

# Store Embedding : ChromaDB

In [33]:
# Add to vector database

embedding=OllamaEmbeddings(model="nomic-embed-text",show_progress=True)
persist_directory='private_docs'

vectordb = Chroma.from_documents(
    documents=chunks, 
    embedding=embedding,
    # collection_name="local-rag",
    persist_directory=persist_directory   
)
 
# vectordb.persist()

OllamaEmbeddings: 100%|██████████| 3/3 [00:05<00:00,  1.75s/it]


# Load Embedding 

In [34]:
embedding=OllamaEmbeddings(model="nomic-embed-text",show_progress=True)
persist_directory='private_docs'

vectordb = Chroma(persist_directory=persist_directory, embedding_function=embedding)

vectordb._collection.count()

3

## Similarity Search

In [44]:
question = "what is my name"
docs = vectordb.similarity_search(question,k=1)
docs

OllamaEmbeddings: 100%|██████████| 1/1 [00:05<00:00,  5.73s/it]


[Document(page_content='Biography of Abu Sufyan\nMy name is Abu Sufyan. I am the son of Zafar Iqbal, and I am 25 years old. I graduated from the\nVirtual University of Pakistan with a degree in Computer Science. During my studies, I specialized in', metadata={'page': 0, 'source': '/home/abusufyan/development/local_llm/data/Abu_Sufyan_Bio.pdf'})]

# Load Ollama Model

In [46]:
# LLM from Ollama
local_model = "mistral"
llm = ChatOllama(model=local_model)
llm

ChatOllama(model='mistral')

# Template

In [47]:
template = """
Crafted by VetandTech, you are a virtual assistant, meticulously designed to provide comprehensive veterinary data and information to 
veterinary professionals only. The data you provide is not for pet parents/owners; it’s only and only for vet professionals/practitioners. 

Your expertise extends to addressing vet-related queries with compassion and precision. 
Your role is to emulate a dedicated professional in the field, offering thorough guidance and solutions for various pet/animals health issues. 


CONTEXT:
{context}

Follow Up Input: 
{question}

CHAT HISTORY: 
{chat_history}
"""

# Initialize the prompt
QA_PROMPT = PromptTemplate(template=template, input_variables=[
                    "question", "context","chat_history"])


In [50]:
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory, ChatMessageHistory

# LLM from Ollama
local_model = "mistral"
llm = ChatOllama(model=local_model)



memory = ConversationBufferMemory(
            memory_key="chat_history",
            return_messages=True)
        
first_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=vectordb.as_retriever(),
    combine_docs_chain_kwargs={"prompt": QA_PROMPT},
    memory=memory,
    )
        
#*********************************************
answer = first_chain({"question": question})
#*********************************************

answer

ValidationError: 1 validation error for LLMChain
llm
  Can't instantiate abstract class BaseLanguageModel with abstract methods agenerate_prompt, apredict, apredict_messages, generate_prompt, invoke, predict, predict_messages (type=type_error)

In [ ]:
i would suggest ye hee apk lyee better 

In [121]:
chain.invoke("what is my first name?")

OllamaEmbeddings: 100%|██████████| 1/1 [00:02<00:00,  2.13s/it]


' Abu Sufyan\n\nExplanation: The context provided in the document mentions that the person introduced in the biography is named Abu Sufyan.'

In [30]:
chain.invoke("tell me about my age?") 

OllamaEmbeddings: 100%|██████████| 1/1 [00:01<00:00,  1.61s/it]
Number of requested results 4 is greater than number of elements in index 3, updating n_results = 3


' Abu Sufyan is 25 years old.'